# Keyword baseline (Issue #8)

**Summary.** A hand-written, rule-based detector for the 10 core CUAD clause categories. Each chunk is scored per category from evidence groups (anchor +2, support +1, counterevidence -1, group-max, floor 0) and a chunk is flagged when its score reaches a per-category threshold. Rules were drafted and audited on the 326 **train** contracts only, frozen and committed (commit `227b133`) before any validation scoring, and thresholds were then chosen on the 82 **validation** contracts, as the spec requires.

- Headline: validation macro-F1 and micro-F1 are shown in the results table below (macro-F1 0.673 at the time of writing).
- The chunks are the provisional 768-token / 512-overlap chunks (`gate3_chunking_v1`).
- F1 on train is optimistic (the phrases were found there) and is only used as a bug check.
- This notebook only loads and checks saved outputs. The scoring itself is `src/keyword_baseline/`.

## 1. Paths and settings (single place)

In [1]:
from pathlib import Path
import hashlib
import json
import subprocess

import pandas as pd

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
RULES = ROOT / "configs" / "keyword_rules_v1.json"
THRESHOLDS = ROOT / "configs" / "keyword_thresholds_v1.json"
FREEZE = ROOT / "configs" / "keyword_freeze_v1.md"
REPORTS = ROOT / "reports" / "keyword_baseline"
VAL_CHUNKS = ROOT / "data" / "gate3" / "validation_chunks_768_512.csv"

RERUN = False  # True re-scores the validation chunks and re-picks thresholds (a few minutes)

## 2. Optional: re-run the pipeline

Run from the repo root. Step 1 reads chunk text only; labels are joined only in step 2.

In [2]:
if RERUN:
    subprocess.run([str(ROOT / ".venv" / "bin" / "python"), "-m", "src.keyword_baseline.score_chunks",
                    "--chunks", str(VAL_CHUNKS), "--out-prefix", str(REPORTS / "validation")], cwd=ROOT, check=True)
    subprocess.run([str(ROOT / ".venv" / "bin" / "python"), "-m", "src.keyword_baseline.evaluate"], cwd=ROOT, check=True)

## 3. The frozen rules are unchanged

In [3]:
def sha256(path):
    return hashlib.sha256(path.read_bytes()).hexdigest()

th = json.loads(THRESHOLDS.read_text())
freeze_text = FREEZE.read_text()
checks = pd.Series({
    "rules file hash equals thresholds file record": sha256(RULES) == th["rules_sha256"],
    "rules file hash appears in the freeze file": sha256(RULES) in freeze_text,
    "predictions hash equals thresholds file record": sha256(REPORTS / "validation_predictions.csv") == th["predictions_sha256"],
})
checks

rules file hash equals thresholds file record     True
rules file hash appears in the freeze file        True
predictions hash equals thresholds file record    True
dtype: bool

## 4. Rule inventory

In [4]:
rules = json.loads(RULES.read_text())
inventory = pd.DataFrame(
    [{"category": cat, "role": g["role"], "group": g["group_id"], "rules": len(g["rules"])}
     for cat, c in rules["categories"].items() for g in c["groups"]]
)
summary = (inventory.pivot_table(index="category", columns="role", values="group", aggfunc="count", fill_value=0)
           .join(inventory.groupby("category").rules.sum().rename("rules"))
           .join(pd.Series({cat: len(c["vetoes"]) for cat, c in rules["categories"].items()}, name="vetoes")))
summary

,anchor,counter,support,rules,vetoes
category,,,,,
Anti-Assignment,3,1,4,18,0
Cap On Liability,1,0,7,20,0
Change Of Control,3,0,4,19,0
Exclusivity,1,0,3,8,6
Expiration Date,3,0,5,22,0
License Grant,2,0,2,12,0
Non-Compete,2,0,7,26,0
Non-Transferable License,3,1,1,16,0
Renewal Term,2,1,2,17,1


## 5. Validation results (unmasked pairs, one threshold per category)

Threshold rule: integers 1 up to the maximum possible score; highest F1, then higher recall, then lower threshold.

In [5]:
metrics = pd.read_csv(REPORTS / "validation_metrics.csv").sort_values("f1", ascending=False)
show = metrics[["category", "threshold", "precision", "recall", "f1", "tp", "fp", "fn", "support", "positive_contracts"]]
display(show.round(3).reset_index(drop=True))
print(f"macro-F1 {metrics.f1.mean():.3f}   micro-F1 {th['validation_micro_f1']:.3f}")

,category,threshold,precision,recall,f1,tp,fp,fn,support,positive_contracts
0,Non-Transferable License,2,0.962,0.865,0.911,77,3,12,89,26
1,Anti-Assignment,2,0.793,0.779,0.786,169,44,48,217,67
2,License Grant,3,0.843,0.700,0.765,145,27,62,207,41
3,Renewal Term,1,0.797,0.705,0.748,55,14,23,78,31
4,Cap On Liability,2,0.879,0.603,0.715,138,19,91,229,50
5,Expiration Date,3,0.839,0.599,0.699,115,22,77,192,71
6,Uncapped Liability,3,0.820,0.485,0.610,32,7,34,66,20
7,Exclusivity,3,0.525,0.622,0.569,74,67,45,119,31
8,Non-Compete,2,0.429,0.542,0.478,39,52,33,72,19
9,Change Of Control,2,0.336,0.671,0.448,49,97,24,73,19


macro-F1 0.673   micro-F1 0.690


### Threshold search (F1 by threshold)

In [6]:
search = pd.read_csv(REPORTS / "validation_threshold_search.csv")
search.pivot(index="category", columns="threshold", values="f1").round(2)

threshold,1,2,3,4,5,6,7,8,9,10,11
category,,,,,,,,,,,
Anti-Assignment,0.52,0.79,0.77,0.66,0.55,0.33,0.19,0.08,0.03,0.03,NaN
Cap On Liability,0.53,0.72,0.65,0.53,0.35,0.17,0.08,0.00,0.00,NaN,NaN
Change Of Control,0.23,0.45,0.43,0.16,0.00,0.00,0.00,0.00,0.00,0.00,NaN
Exclusivity,0.31,0.46,0.57,0.21,0.00,NaN,NaN,NaN,NaN,NaN,NaN
Expiration Date,0.36,0.60,0.70,0.54,0.44,0.27,0.14,0.03,0.00,0.00,0.0
License Grant,0.60,0.66,0.77,0.71,0.55,0.17,NaN,NaN,NaN,NaN,NaN
Non-Compete,0.20,0.48,0.42,0.38,0.08,0.00,0.00,0.00,0.00,0.00,0.0
Non-Transferable License,0.50,0.91,0.61,0.27,0.07,0.00,0.00,NaN,NaN,NaN,NaN
Renewal Term,0.75,0.71,0.48,0.19,0.05,0.00,NaN,NaN,NaN,NaN,NaN


## 6. Where the baseline struggles

Per-category false-positive and false-negative patterns, with examples, are in `reports/keyword_baseline/weak_behavior.md`. Short version:

- Overlapping categories cause many false positives (Change Of Control vs Anti-Assignment, Uncapped vs Cap On Liability).
- Renewal Term, Cap On Liability and Expiration Date lose recall on paraphrases and fragments the phrases do not cover.
- Non-Compete, Change Of Control and Uncapped Liability have about 20 or fewer positive validation contracts, so their scores are noisy.
- No rule was changed after seeing validation results.

## 7. Artifacts

| File | Content |
|---|---|
| `configs/keyword_rules_v1.json` | Frozen rules (184 IDs) |
| `configs/keyword_freeze_v1.md` | Hashes recorded at freeze |
| `configs/keyword_thresholds_v1.json` | Chosen thresholds and metrics summary |
| `reports/keyword_baseline/decision_tables/` | Per-category decision tables |
| `reports/keyword_baseline/validation_predictions.csv` | 24,810 rows, no labels |
| `reports/keyword_baseline/validation_match_evidence.csv` | Every match with source offsets |
| `reports/keyword_baseline/validation_metrics.csv`, `validation_threshold_search.csv` | Metrics and sweep |
| `reports/keyword_baseline/weak_behavior.md` | Error analysis |